# Settlement Prediction of Braced Excavations — ML Pipeline
**Physics-Informed Machine Learning · Stratified Dhaka Soils · PLAXIS 2D (80 runs)**

Models: **Linear Regression · Polynomial-2 + Ridge · XGBoost**
Targets: **δv,max** (vertical settlement) and **δh,max** (horizontal wall deflection)

This notebook is built to *avoid overfitting* on the small (n=80) dataset:
- Repeated K-Fold CV (5×10) as the headline metric — not a single split
- Scaling fit inside CV folds only (no data leakage)
- Ridge penalty (α tuned inside CV) on the 20-term polynomial
- Shallow, subsampled XGBoost trees
- Explicit train-vs-CV R² gap printed as an overfit alarm
- Minimal feature engineering by design (extra features hurt at n=80)


In [ ]:
# === Setup ===
# If a package is missing, run once:  !pip install xgboost scikit-learn pandas matplotlib seaborn openpyxl
import os, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RepeatedKFold, cross_validate, cross_val_predict, GridSearchCV, train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.inspection import permutation_importance
try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except Exception:
    from sklearn.ensemble import GradientBoostingRegressor
    HAS_XGB = False
    print("xgboost not found -> using GradientBoosting as stand-in. Run: pip install xgboost")

RS = 42
sns.set_style('whitegrid'); plt.rcParams['figure.dpi']=110
np.random.seed(RS)

## 1. Paths & output folder
Edit `DATA_DIR`/`FNAME` if your file name differs. Figures are auto-saved to a **new** `figures/` folder.

In [ ]:
# --- file location (your folder) ---
DATA_DIR = r"C:\thesis work"
FNAME    = "PLAXIS_Complete_80runs.xlsx"
XLSX = os.path.join(DATA_DIR, FNAME)

# --- create a new folder for outputs when you run ---
FIGDIR = os.path.join(DATA_DIR, "figures")
os.makedirs(FIGDIR, exist_ok=True)
def savefig(name):
    p = os.path.join(FIGDIR, name)
    plt.savefig(p, dpi=300, bbox_inches='tight'); print("saved:", p)
print("Reading:", XLSX)
print("Figures will be saved in:", FIGDIR)

## 2. Load data (sheet `All Runs (80)`, header on row 3)

In [ ]:
raw = pd.read_excel(XLSX, sheet_name="All Runs (80)", header=2)
cols = list(raw.columns)
ren = {cols[0]:'Run', cols[1]:'Status', cols[2]:'E50_SC', cols[5]:'phi',
       cols[6]:'E50_MAD', cols[9]:'WT', cols[10]:'Thick', cols[11]:'dv', cols[12]:'dh'}
df = raw.rename(columns=ren)
df = df[pd.to_numeric(df['Run'], errors='coerce').notna()].copy()
for c in ['E50_SC','phi','Thick','E50_MAD','WT','dv','dh']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df = df.dropna(subset=['dv','dh']).reset_index(drop=True)

FEATURES = ['E50_SC','phi','Thick','E50_MAD','WT']
LABELS = {'E50_SC':"E50 Soft Clay (kN/m2)",'phi':"phi' Soft Clay (deg)",
          'Thick':"Soft Clay Thickness (m)",'E50_MAD':"E50 Madhupur (kN/m2)",'WT':"Water Table Depth (m)"}
print("Rows:", len(df))
df[FEATURES+['dv','dh']].describe().round(2)

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# 3a. Correlation heatmap
corr = df[FEATURES+['dv','dh']].corr()
plt.figure(figsize=(7,6))
sns.heatmap(corr, annot=True, fmt='+.2f', cmap='RdBu_r', center=0, square=True, cbar_kws={'label':'Pearson r'})
plt.title('Correlation Matrix - Inputs vs Outputs')
savefig('01_correlation_heatmap.png'); plt.show()

In [ ]:
# 3b. Scatter plots: each feature vs dv and dh
fig, axes = plt.subplots(2, 5, figsize=(20,7))
for j,f in enumerate(FEATURES):
    for i,(tgt,lab) in enumerate([('dv','dv,max (mm)'),('dh','dh,max (mm)')]):
        ax=axes[i,j]
        ax.scatter(df[f], df[tgt], s=28, alpha=0.7, edgecolor='k', linewidth=0.3)
        ax.set_xlabel(LABELS[f])
        if j==0: ax.set_ylabel(lab)
fig.suptitle('Parameter vs Output Scatter Plots', y=1.02, fontsize=14)
plt.tight_layout(); savefig('02_scatter_features.png'); plt.show()

In [ ]:
# 3c. Distributions
fig, axes = plt.subplots(2, 4, figsize=(18,8))
for ax,c in zip(axes.ravel(), FEATURES+['dv','dh']):
    sns.histplot(df[c], kde=True, ax=ax, color='steelblue'); ax.set_title(LABELS.get(c,c))
axes.ravel()[-1].axis('off')
fig.suptitle('Distributions', y=1.01, fontsize=14)
plt.tight_layout(); savefig('03_distributions.png'); plt.show()

In [ ]:
# 3d. Box plots (standardized)
z = (df[FEATURES]-df[FEATURES].mean())/df[FEATURES].std()
plt.figure(figsize=(9,5))
sns.boxplot(data=z, palette='Set2')
plt.xticks(range(len(FEATURES)), [LABELS[f] for f in FEATURES], rotation=20, ha='right')
plt.ylabel('Standardized value (z)'); plt.title('Input Spread (standardized)')
savefig('04_boxplots.png'); plt.show()

## 4. Model definitions
Pipelines keep scaling **inside** CV (no leakage). XGBoost gets no scaling (trees don't need it).
The polynomial's Ridge alpha is tuned by an inner search to control overfitting.

In [ ]:
X = df[FEATURES].values
CV = RepeatedKFold(n_splits=5, n_repeats=10, random_state=RS)

def make_linear():
    return Pipeline([('scaler',StandardScaler()), ('model',LinearRegression())])

def make_poly_ridge():
    pipe = Pipeline([('scaler',StandardScaler()),
                     ('poly',PolynomialFeatures(degree=2, include_bias=False)),
                     ('model',Ridge())])
    grid = {'model__alpha':[1,5,10,20,50,100]}
    return GridSearchCV(pipe, grid, cv=5, scoring='r2')

def make_booster():
    if HAS_XGB:
        return XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.05,
                            subsample=0.9, colsample_bytree=0.9, random_state=RS, verbosity=0)
    return GradientBoostingRegressor(n_estimators=300, max_depth=3, learning_rate=0.05,
                                     subsample=0.9, random_state=RS)

BOOST_NAME = 'XGBoost' if HAS_XGB else 'GradientBoosting'
MODELS = {'Linear Regression':make_linear, 'Poly-2 + Ridge':make_poly_ridge, BOOST_NAME:make_booster}

## 5. Train + evaluate (repeated CV) with overfit check

In [ ]:
def evaluate(target):
    y = df[target].values; rows=[]
    for name, mk in MODELS.items():
        sc = cross_validate(mk(), X, y, cv=CV,
              scoring=['r2','neg_root_mean_squared_error','neg_mean_absolute_error'],
              return_train_score=True)
        rows.append({'Model':name,
            'CV R2':sc['test_r2'].mean(), '+/-std':sc['test_r2'].std(),
            'Train R2':sc['train_r2'].mean(),
            'Overfit gap':sc['train_r2'].mean()-sc['test_r2'].mean(),
            'RMSE':-sc['test_neg_root_mean_squared_error'].mean(),
            'MAE':-sc['test_neg_mean_absolute_error'].mean()})
    t = pd.DataFrame(rows).set_index('Model')
    print(f"\n=== {target} ===  (Overfit gap > ~0.10 = warning)")
    return t.round(3)

res_dv = evaluate('dv'); display(res_dv)
res_dh = evaluate('dh'); display(res_dh)

## 6. Model comparison table (saved as CSV)

In [ ]:
comp = pd.concat({'dv,max':res_dv, 'dh,max':res_dh}, axis=0)
comp.to_csv(os.path.join(FIGDIR,'model_comparison.csv'))
print('saved:', os.path.join(FIGDIR,'model_comparison.csv'))
comp

## 7. Predicted vs Actual (parity) + Residual plots
Out-of-fold predictions — honest (each point predicted while held out).

In [ ]:
def parity_and_resid(target):
    from sklearn.model_selection import KFold
    oof_cv = KFold(n_splits=5, shuffle=True, random_state=RS)  # single partition for OOF preds
    y = df[target].values
    fig, axes = plt.subplots(2, len(MODELS), figsize=(6*len(MODELS),10))
    for j,(name,mk) in enumerate(MODELS.items()):
        pred = cross_val_predict(mk(), X, y, cv=oof_cv)
        r2=r2_score(y,pred); rmse=mean_squared_error(y,pred)**0.5
        ax=axes[0,j]; lim=[min(y.min(),pred.min())*0.95, max(y.max(),pred.max())*1.05]
        ax.scatter(y,pred,s=28,alpha=0.7,edgecolor='k',linewidth=0.3)
        ax.plot(lim,lim,'r--'); ax.set_xlim(lim); ax.set_ylim(lim)
        ax.set_title(f'{name}\nR2={r2:.3f}  RMSE={rmse:.2f} mm')
        ax.set_xlabel(f'Actual {target} (mm)')
        if j==0: ax.set_ylabel(f'Predicted {target} (mm)')
        ax2=axes[1,j]; resid=y-pred
        ax2.scatter(pred,resid,s=28,alpha=0.7,edgecolor='k',linewidth=0.3)
        ax2.axhline(0,color='r',ls='--'); ax2.set_xlabel(f'Predicted {target} (mm)')
        if j==0: ax2.set_ylabel('Residual (mm)')
    fig.suptitle(f'{target}: Parity (top) & Residuals (bottom)', y=1.01, fontsize=14)
    plt.tight_layout(); savefig(f'05_parity_residual_{target}.png'); plt.show()

parity_and_resid('dv'); parity_and_resid('dh')

## 8. Feature importance
Permutation importance (model-agnostic) for the booster; standardized coefficients for the linear model.

In [ ]:
def feat_importance(target):
    y=df[target].values
    Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.25,random_state=RS)
    fig,axes=plt.subplots(1,2,figsize=(14,5))
    bm=MODELS[BOOST_NAME](); bm.fit(Xtr,ytr)
    pi=permutation_importance(bm,Xte,yte,n_repeats=30,random_state=RS)
    order=np.argsort(pi.importances_mean)
    axes[0].barh([LABELS[FEATURES[i]] for i in order], pi.importances_mean[order],
                 xerr=pi.importances_std[order], color='teal')
    axes[0].set_title(f'{BOOST_NAME} - Permutation Importance ({target})')
    lm=make_linear(); lm.fit(X,y); coef=lm.named_steps['model'].coef_
    o2=np.argsort(np.abs(coef))
    axes[1].barh([LABELS[FEATURES[i]] for i in o2], coef[o2], color='indianred')
    axes[1].axvline(0,color='k',lw=0.8); axes[1].set_title(f'Linear - Std Coefficients ({target})')
    plt.tight_layout(); savefig(f'06_feature_importance_{target}.png'); plt.show()

feat_importance('dv'); feat_importance('dh')

## 9. Sensitivity analysis — 5 curves per output
One parameter swept across its range; others held at dataset median. Uses trained **Poly-2 + Ridge**.

In [ ]:
def sensitivity(target):
    y=df[target].values; model=make_poly_ridge(); model.fit(X,y)
    med=df[FEATURES].median().values
    fig,axes=plt.subplots(1,5,figsize=(22,4))
    for j,f in enumerate(FEATURES):
        grid=np.linspace(df[f].min(),df[f].max(),50)
        Xs=np.tile(med,(50,1)); Xs[:,j]=grid
        axes[j].plot(grid, model.predict(Xs), lw=2, color='darkblue')
        axes[j].scatter(df[f],y,s=14,alpha=0.3,color='gray')
        axes[j].set_xlabel(LABELS[f])
        if j==0: axes[j].set_ylabel(f'{target} (mm)')
    fig.suptitle(f'Sensitivity of {target} (others at median)', y=1.03, fontsize=14)
    plt.tight_layout(); savefig(f'07_sensitivity_{target}.png'); plt.show()

sensitivity('dv'); sensitivity('dh')

## 10. Governing equation (Poly-2 + Ridge)
Top standardized polynomial terms printed and saved. Feature means/stds saved so you can
convert to raw units if needed.

In [ ]:
def equation(target):
    y=df[target].values; gs=make_poly_ridge(); gs.fit(X,y); best=gs.best_estimator_
    poly=best.named_steps['poly']; ridge=best.named_steps['model']
    names=poly.get_feature_names_out(FEATURES)
    terms=sorted(zip(names, ridge.coef_), key=lambda t:-abs(t[1]))
    lines=[f"{target},max  (Poly-2 + Ridge, alpha={ridge.alpha})",
           f"  intercept = {ridge.intercept_:.4f}",
           "  (coefficients on standardized features z=(x-mean)/std; top terms)"]
    for n,c in terms[:12]:
        if abs(c)>1e-6: lines.append(f"    {c:+.4f} * {n}")
    txt="\n".join(lines); print(txt)
    with open(os.path.join(FIGDIR,f'equation_{target}.txt'),'w') as fp:
        fp.write(txt+"\n\nmeans:\n"+str(df[FEATURES].mean().to_dict())+"\n\nstds:\n"+str(df[FEATURES].std().to_dict()))
    print("saved:", os.path.join(FIGDIR,f'equation_{target}.txt'))

equation('dv'); print(); equation('dh')

## 11. Reduced governing equation (10-term, raw units) — RECOMMENDED

The full degree-2 polynomial has 21 terms; many are near-zero and capture noise.
A **reduced 10-term model** (dominant linear, square, and interaction terms) generalises
*better* on this small dataset:

| Model | δv CV R² | δh CV R² |
|---|---|---|
| Full Poly-2 + Ridge (21 terms) | 0.931 | 0.916 |
| **Reduced (10 terms)** | **0.954** | **0.957** |

Coefficients are fitted by ordinary least squares directly on **raw engineering units**
(no scaling), so the equation can be used as-is. The cell below refits, prints, verifies
CV R², and saves the equations to `figures/reduced_equation.txt`.

**Variable key:** `E50sc`=E50 Soft Clay (kN/m²), `phi`=φ′ (°), `H`=Soft Clay Thickness (m),
`E50md`=E50 Madhupur (kN/m²), `WT`=Water Table Depth (m). Output in mm.


In [ ]:
# === Reduced 10-term equation in raw units ===
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

_idx = {f:i for i,f in enumerate(FEATURES)}
# dominant term set: 5 linear + H^2, WT^2, H*WT, E50sc*H, E50sc*WT
REDUCED_TERMS = [(_idx['E50_SC'],),(_idx['phi'],),(_idx['Thick'],),(_idx['E50_MAD'],),(_idx['WT'],),
                 (_idx['Thick'],_idx['Thick']),(_idx['WT'],_idx['WT']),
                 (_idx['Thick'],_idx['WT']),(_idx['E50_SC'],_idx['Thick']),(_idx['E50_SC'],_idx['WT'])]
_LBL = {'E50_SC':'E50sc','phi':'phi','Thick':'H','E50_MAD':'E50md','WT':'WT'}

def _design(terms):
    M=[]
    for t in terms:
        M.append(X[:,t[0]] if len(t)==1 else X[:,t[0]]*X[:,t[1]])
    return np.column_stack(M)

def _tname(t):
    if len(t)==1: return _LBL[FEATURES[t[0]]]
    i,j=t
    return f"{_LBL[FEATURES[i]]}^2" if i==j else f"{_LBL[FEATURES[i]]}*{_LBL[FEATURES[j]]}"

def reduced_equation(target):
    y = df[target].values
    D = _design(REDUCED_TERMS)
    lr = LinearRegression().fit(D, y)
    r2 = cross_val_score(lr, D, y, cv=CV, scoring='r2').mean()
    lines = [f"{target},max   (reduced 10-term, raw units)   CV R2 = {r2:.3f}",
             f"  intercept = {lr.intercept_:+.5f}"]
    for c,t in zip(lr.coef_, REDUCED_TERMS):
        lines.append(f"    {c:+.6e} * {_tname(t)}")
    txt = "\n".join(lines)
    print(txt); print()
    return txt, r2

_all=[]
for tgt in ['dv','dh']:
    txt,_ = reduced_equation(tgt); _all.append(txt)
with open(os.path.join(FIGDIR,'reduced_equation.txt'),'w') as fp:
    fp.write("\n\n".join(_all))
print("saved:", os.path.join(FIGDIR,'reduced_equation.txt'))

---
### Notes for your thesis
- **Primary model:** Poly-2 + Ridge (highest CV R2, lowest error, gives an equation).
- **Overfit control:** repeated CV, in-fold scaling, Ridge alpha tuned in CV, shallow trees, train-vs-CV gap printed.
- **No extra feature engineering** by design - at n=80 it adds variance, not signal.
- All figures are 300 dpi in the `figures/` folder; comparison table and equations saved as files.
